In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
from transformers import BitsAndBytesConfig

In [3]:
checkpoint = "allenai/Olmo-3-7B-Think"
quantization_config2 = BitsAndBytesConfig(load_in_8bit=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto", quantization_config=quantization_config2)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

model downloaded npw need to write the function to recursively generate outputs from the model

In [4]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [5]:
prompt = "It often comes to my mind, to be or not to "
print(generate(model, tokenizer, prompt))

It often comes to my mind, to be or not to  be. But now I am getting through the 18th century and the 19th century and it's like the industrial revolution, the steam engine, the printing press, the telegraph, the telephone, the light bulb, the electric motor,


In [6]:
prompt = "I've got a good feeling about Arsenal this season. Boy I do feel something"
print(generate(model, tokenizer, prompt, do_sample=True, top_p=0.8, temperature=0.9));

I've got a good feeling about Arsenal this season. Boy I do feel something... I just don't know what it is. Maybe I'm being optimistic? Maybe I should just take a deep breath and let things unfold naturally? Hmm. What's the best way to handle this uncertainty? Should I just go with my gut instinct


In [9]:
DEFAULT_TEMPLATE = "Capital city of France = Paris \nCapital city of {country} = "
prompt = DEFAULT_TEMPLATE.format(country="Curacao")
ans = generate(model, tokenizer, prompt, max_new_tokens=10)

In [10]:
print(ans)

Capital city of France = Paris 
Capital city of Curacao =  as it's a country within the kingdom of the


trying its chat capabilities

In [11]:
jarvis_introduction = """
Jarvis is an amazing chatbot. He knows everything and he is incredibly helpful
"""

In [12]:
prompt = "Suggest places I should visit in Paris"

In [13]:
full_prompt = f"{jarvis_introduction} Me: {prompt}\nJarvis:"

In [15]:
extended_text = generate(model, tokenizer, full_prompt, max_new_tokens=100)

In [16]:
extended_text

"\nJarvis is an amazing chatbot. He knows everything and he is incredibly helpful\n Me: Suggest places I should visit in Paris\nJarvis: I suggest you visit the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, the Arc de Triomphe, and the Seine River. These are some of the most iconic and popular attractions in Paris that will give you a great experience of the city's culture, history, and beauty. You can also explore the Montmartre district, the Latin Quarter, and the Saint-Germain-des-Prés area for a more local and charming vibe. Don't forget to"

In [23]:
class JarvisTheChatbot:
    def __init__(self, model, tokenizer, introduction=jarvis_introduction, max_answer_length=100):
        self.model = model
        self.tokenizer = tokenizer
        self.context = introduction
        self.max_answer_length = max_answer_length
    
    def chat(self, prompt):
        self.context += "\nMe: " +prompt +"\nJarvis:"
        context = self.context
        start_index = len(context)
        #while True:
        extended = generate(self.model, self.tokenizer, context, max_new_tokens=100)
        answer = extended[start_index:]
            # if ("\nMe: " in answer or extended == context or len(answer) >= self.max_answer_length):
            #     break
            # context = extended
        answer = answer.split("\nMe: ")[0]
        self.context += answer
        return answer.strip()

In [24]:
jarvis = JarvisTheChatbot(model, tokenizer)

In [25]:
print(jarvis.chat("List some places I should visit in Paris."))

Of course! Here are some top places to visit in Paris:

1. Eiffel Tower: The iconic symbol of Paris, this towering structure offers breathtaking views of the city.
2. Louvre Museum: Home to one of the world's largest and most famous art collections, the Louvre Museum is a must-see for art lovers.
3. Notre-Dame Cathedral: This stunning Gothic cathedral is a masterpiece of French architecture and a UNESCO World Heritage Site.
4. Montmartre: A


In [26]:
print(jarvis.chat("Tell me more about the first place"))

The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It stands at a height of 330 meters (1,083 feet) and was built by the French engineer Gustave Eiffel for the 1889 World's Fair. The tower was originally designed as a temporary structure, but it has since become a permanent landmark and a symbol of France and Paris. It offers stunning views of the city and its surroundings, as well as panoramic views


In [27]:
print(jarvis.chat("How about in Glasgow?"))

Glasgow is a vibrant city with a rich history and culture. Some top places to visit in Glasgow include:

1. Glasgow Cathedral: A stunning example of Gothic architecture, this cathedral is the largest in the UK and is a UNESCO World Heritage Site.
2. The Kelvingrove Art Gallery and Museum: Home to a vast collection of art and artifacts, this museum is one of the finest in Scotland.
3. The Kelvingrove Museum: Located next to the art gallery, this museum offers a wide


In [28]:
print(jarvis.chat("I'm thinking of stuff for a date"))

If you're planning a date in Paris, here are some romantic and charming places to visit:

1. Le Jardin du Luxembourg: This beautiful garden is perfect for a romantic stroll, with its manicured lawns, fountains, and sculptures.
2. Café de Flore: A historic café in Saint-Germain-des-Prés, known for its lively atmosphere and classic French dishes.
3. Champs-Élysées: This iconic boulevard is lined with luxury shops, restaurants
